# Data Preparation

Compute or add the following data to S&P500 panel data.
*   Target Variable: realized volatility in the following H days
*   Market data: market return, VIX
*   Industry category



In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
path = "/content/drive/MyDrive/sp500_vol/sp500_panel.parquet"
out_dir = '/content/drive/MyDrive/sp500_vol'

H = 5
START, END = "2019-01-01", "2025-12-31"

panel = pd.read_parquet(path)
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values(['permno','date']).reset_index(drop= True)
print(panel.shape, panel.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1039122 entries, 0 to 1039121
Data columns (total 8 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   permno      1039122 non-null  Int64         
 1   date        1039122 non-null  datetime64[ns]
 2   return      1038816 non-null  Float64       
 3   price       1038861 non-null  Float64       
 4   volumn      1038802 non-null  Float64       
 5   isin_sp500  1039122 non-null  bool          
 6   log_return  1038816 non-null  Float64       
 7   split       1039122 non-null  object        
dtypes: Float64(4), Int64(1), bool(1), datetime64[ns](1), object(1)
memory usage: 61.4+ MB
(1039122, 8) None


In [4]:
print(panel["date"].max(),panel["date"].min())

2025-12-31 00:00:00 2019-01-02 00:00:00


## **Realized volatility for the following H days (closing price, annualize)**

In [5]:
#Compute target variable
panel["r2"] = panel["log_return"] ** 2

In [6]:
panel.head()

,permno,date,return,price,volumn,isin_sp500,log_return,split,r2
0,10104,2019-01-02,0.00155,45.22,14320441.0,True,0.001549,warmup,0.000002
1,10104,2019-01-03,-0.00973,44.78,19868713.0,True,-0.009778,warmup,0.000096
2,10104,2019-01-04,0.0431,46.71,20983953.0,True,0.042197,warmup,0.001781
3,10104,2019-01-07,0.015842,47.45,17967907.0,True,0.015718,warmup,0.000247
4,10104,2019-01-08,0.009062,47.88,16255688.0,True,0.009021,warmup,0.000081


In [7]:
g = panel.groupby('permno')['r2']
future_r2 = pd.concat([g.shift(-i) for i in range(1,H+1)], axis=1)
panel["future_r2_sum"] = future_r2.sum(axis=1, min_count=1)
panel["n_future_r2"] = future_r2.count(axis = 1)
panel["ann_future_realized_volatility"] = np.sqrt(252 * panel["future_r2_sum"] / panel["n_future_r2"])

panel.loc[panel["n_future_r2"].fillna(0) == 0, "ann_future_realized_volatility"] = np.nan
panel["full_r2"] = panel["n_future_r2"] == H
print(panel.head())
print("without full data ratio", (~panel.loc[panel["isin_sp500"],"full_r2"]).mean())

   permno       date    return  price      volumn  isin_sp500  log_return  \
0   10104 2019-01-02   0.00155  45.22  14320441.0        True    0.001549   
1   10104 2019-01-03  -0.00973  44.78  19868713.0        True   -0.009778   
2   10104 2019-01-04    0.0431  46.71  20983953.0        True    0.042197   
3   10104 2019-01-07  0.015842  47.45  17967907.0        True    0.015718   
4   10104 2019-01-08  0.009062  47.88  16255688.0        True    0.009021   

    split        r2  future_r2_sum  n_future_r2  \
0  warmup  0.000002       0.002209            5   
1  warmup  0.000096       0.002114            5   
2  warmup  0.001781       0.000429            5   
3  warmup  0.000247       0.000187            5   
4  warmup  0.000081       0.000171            5   

   ann_future_realized_volatility  full_r2  
0                        0.333667     True  
1                        0.326421     True  
2                        0.147073     True  
3                        0.097166     True  
4    

In [8]:
#Market return
mkt = panel.loc[panel["isin_sp500"] == True]
mkt_avg_return_bydate = mkt.groupby("date")["return"].mean()
mkt_avg_return_bydate.head()
mkt_avg_return_bydate = mkt_avg_return_bydate.rename("market_daily_return").to_frame()
mkt_log_ret = np.log1p(mkt_avg_return_bydate)

In [9]:
panel = panel.merge(mkt_log_ret,left_on= 'date', right_index= True, how= 'left')
panel.head()

,permno,date,return,price,volumn,isin_sp500,log_return,split,r2,future_r2_sum,n_future_r2,ann_future_realized_volatility,full_r2,market_daily_return
0,10104,2019-01-02,0.00155,45.22,14320441.0,True,0.001549,warmup,0.000002,0.002209,5,0.333667,True,0.000757
1,10104,2019-01-03,-0.00973,44.78,19868713.0,True,-0.009778,warmup,0.000096,0.002114,5,0.326421,True,-0.019026
2,10104,2019-01-04,0.0431,46.71,20983953.0,True,0.042197,warmup,0.001781,0.000429,5,0.147073,True,0.032936
3,10104,2019-01-07,0.015842,47.45,17967907.0,True,0.015718,warmup,0.000247,0.000187,5,0.097166,True,0.010268
4,10104,2019-01-08,0.009062,47.88,16255688.0,True,0.009021,warmup,0.000081,0.000171,5,0.09282,True,0.010726


In [10]:
#realized volatitlity for each stock in the past 5 and 22 days
past_5_ann_r2 = np.sqrt(252* panel.groupby("permno")['r2'].rolling(5, min_periods = 5).mean()).reset_index(level = 0, drop= True)
past_22_ann_r2 = np.sqrt(252* panel.groupby("permno")['r2'].rolling(22, min_periods = 22).mean()).reset_index(level = 0, drop= True)
panel["past_5days_ann_realized_volatility"] = past_5_ann_r2
panel["past_22days_ann_realized_volatility"] = past_22_ann_r2
panel.head()

,permno,date,return,price,volumn,isin_sp500,log_return,split,r2,future_r2_sum,n_future_r2,ann_future_realized_volatility,full_r2,market_daily_return,past_5days_ann_realized_volatility,past_22days_ann_realized_volatility
0,10104,2019-01-02,0.00155,45.22,14320441.0,True,0.001549,warmup,0.000002,0.002209,5,0.333667,True,0.000757,NaN,NaN
1,10104,2019-01-03,-0.00973,44.78,19868713.0,True,-0.009778,warmup,0.000096,0.002114,5,0.326421,True,-0.019026,NaN,NaN
2,10104,2019-01-04,0.0431,46.71,20983953.0,True,0.042197,warmup,0.001781,0.000429,5,0.147073,True,0.032936,NaN,NaN
3,10104,2019-01-07,0.015842,47.45,17967907.0,True,0.015718,warmup,0.000247,0.000187,5,0.097166,True,0.010268,NaN,NaN
4,10104,2019-01-08,0.009062,47.88,16255688.0,True,0.009021,warmup,0.000081,0.000171,5,0.09282,True,0.010726,0.333518,NaN


In [16]:
#VIX data
sid = 'VIXCLS'
url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}"
vix_dt = pd.read_csv(url)
vix_dt.head()
vix_dt["observation_date"] = pd.to_datetime(vix_dt["observation_date"])
vix_dt['VIXCLS'] = pd.to_numeric(vix_dt['VIXCLS'])
vix_dt = vix_dt[(vix_dt["observation_date"] >= '2019-01-01') & (vix_dt["observation_date"] <= '2025-12-31')]

In [19]:
trading_days = pd.DatetimeIndex(sorted(panel["date"].unique()))
vix_dt = vix_dt.reindex(vix_dt.index.union(trading_days)).ffill(limit= 5).reindex(trading_days)

In [20]:
vix_dt.head()

,observation_date,VIXCLS
2019-01-02,2025-12-31,14.95
2019-01-03,2025-12-31,14.95
2019-01-04,2025-12-31,14.95
2019-01-07,2025-12-31,14.95
2019-01-08,2025-12-31,14.95


In [21]:
print(panel.index.equals(vix_dt.index))

False


In [22]:
panel.index

RangeIndex(start=0, stop=1039122, step=1)

In [23]:
panel = panel.merge(vix_dt, left_on='date', right_index= True, how= 'left')
panel.head()

,permno,date,return,price,volumn,isin_sp500,log_return,split,r2,future_r2_sum,n_future_r2,ann_future_realized_volatility,full_r2,market_daily_return,past_5days_ann_realized_volatility,past_22days_ann_realized_volatility,observation_date,VIXCLS
0,10104,2019-01-02,0.00155,45.22,14320441.0,True,0.001549,warmup,0.000002,0.002209,5,0.333667,True,0.000757,NaN,NaN,2025-12-31,14.95
1,10104,2019-01-03,-0.00973,44.78,19868713.0,True,-0.009778,warmup,0.000096,0.002114,5,0.326421,True,-0.019026,NaN,NaN,2025-12-31,14.95
2,10104,2019-01-04,0.0431,46.71,20983953.0,True,0.042197,warmup,0.001781,0.000429,5,0.147073,True,0.032936,NaN,NaN,2025-12-31,14.95
3,10104,2019-01-07,0.015842,47.45,17967907.0,True,0.015718,warmup,0.000247,0.000187,5,0.097166,True,0.010268,NaN,NaN,2025-12-31,14.95
4,10104,2019-01-08,0.009062,47.88,16255688.0,True,0.009021,warmup,0.000081,0.000171,5,0.09282,True,0.010726,0.333518,NaN,2025-12-31,14.95


In [24]:
panel.to_parquet(f"{out_dir}/sp500_panel_features.parquet", index=False)